# Data Science Job Market Analysis

**Name:** [Your Name]  
**Date:** [Current Date]  
**Dataset:** Data Science Job Market Trends


## Project Overview
This notebook provides a comprehensive analysis of data science job market trends, skill requirements, and career pathways using data-driven approaches. The analysis focuses on identifying key skills, emerging trends, and personalized recommendations for career development.

## 1. Initial Setup and Configuration

### 1.1 Import Required Libraries
Import all necessary packages and configure the environment for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import warnings
from tqdm.auto import tqdm
import sys
warnings.filterwarnings('ignore')

import plotly.io as pio

# Set up notebook display
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
%matplotlib inline
pio.renderers.default = 'notebook'  # For Plotly in notebook

# Set up plotting style
plt.style.use('default')
sns.set_palette("viridis")

In [ ]:
sys.path.append('..')
from src.analyze_jobs import DataScienceJobsAnalyzer

# Initialize the analyzer
analyzer = DataScienceJobsAnalyzer(data_dir="../data")

### 1.2 Load and Prepare Data
Load the pre-processed dataset containing job market information.

In [ ]:
# Load cleaned data and create skill pivot
try:
    df = analyzer.load_cleaned_data()
    pivot_df = analyzer.create_skill_pivot(df)
    
    print(f"✅ Successfully loaded {len(df):,} job records")
    print(f"📊 Created skill pivot with {len(pivot_df):,} skill mentions")
    print(f"🎯 Analyzing {pivot_df['skill'].nunique():,} unique skills")
    
    # Display basic dataset info
    print("\n📈 Dataset Overview:")
    print(f"   • Time period: {pivot_df['date'].min()} to {pivot_df['date'].max()}")
    print(f"   • Countries: {pivot_df['country'].nunique()}")
    print(f"   • Companies: {pivot_df['company'].nunique()}")
    print(f"   • Job roles: {pivot_df['cleaned_title_category'].nunique()}")
    print(f"   • Seniority levels: {pivot_df['seniority_level'].nunique()}")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")

## 2. Market Overview Analysis
### 2.1 Overall Skill Frequency Analysis

In [ ]:
# Analyze top skills across the entire market
top_skills = analyzer.analyze_skill_frequency(pivot_df, top_n=5)

print("🏆 TOP 5 MOST IN-DEMAND SKILLS")
print("="*50)

plt.figure(figsize=(14, 8))
analyzer.plot_skills_bar(top_skills, column='skill', top_n=5, 
                        title='Top 5 Most In-Demand Data Science Skills')

# Display detailed table
display(top_skills.head(5).style.background_gradient(cmap='Blues', subset=['prevalence'])
       .format({'prevalence': '{:.1f}%', 'mentions': '{:,}'})
       .set_caption('Top 5 Skills by Prevalence'))

## 3. Seniority-Level Analysis
### 3.1 Skills by Seniority Level

In [ ]:
# Analyze skills across different seniority levels
seniority_skills = analyzer.analyze_skills_by_seniority(pivot_df, top_n=5)

print("👨‍💼 SKILL DISTRIBUTION BY SENIORITY LEVEL")
print("="*55)

# Create visualization for each seniority level
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, (level, skills_df) in enumerate(seniority_skills.items()):
    if idx < 6:  # Limit to first 6 seniority levels
        ax = axes[idx]
        top_skills = skills_df.head(5)
        
        ax.barh(top_skills['skill'], top_skills['prevalence'], color=analyzer.colors[idx])
        ax.set_title(f'{level} Level', fontweight='bold', fontsize=12)
        ax.set_xlabel('Prevalence (%)')
        
        # Add value labels
        for i, v in enumerate(top_skills['prevalence']):
            ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.suptitle('Top Skills by Seniority Level', fontsize=16, fontweight='bold', y=1.02)
plt.show()

### 3.2 Seniority-Specific Skill Patterns

In [ ]:
# Identify seniority-specific patterns
print("🎯 ANALYZING SENIORITY-SPECIFIC SKILL PATTERNS")
print("="*55)

# Get progression data for key skills
key_skills = ['Python', 'SQL', 'Machine Learning', 'AWS', 'Tableau', 'Spark']
progression_data = analyzer.analyze_skill_progression_data(pivot_df, key_skills)

# Analyze patterns
seniority_patterns = analyzer.analyze_seniority_specific_patterns(pivot_df, progression_data)

# Create visualization
analyzer.plot_seniority_patterns(seniority_patterns)

### 3.3 Skill Progression Across Career Levels

In [ ]:
print("📈 SKILL PROGRESSION ACROSS CAREER LEVELS")
print("="*50)

# Plot organized progression view
analyzer.plot_skill_progression_organized(progression_data)

# Additional simple view for key technical skills
technical_skills = ['Python', 'Machine Learning', 'Spark', 'AWS', 'TensorFlow', 'Kubernetes']
tech_progression = analyzer.analyze_skill_progression_data(pivot_df, technical_skills)

print("\n🔧 Technical Skill Progression:")
analyzer.plot_skill_progression_simple(tech_progression, technical_skills)

## 4. Role-Based Analysis
### 4.1 Skills by Job Role 

In [ ]:
# Analyze skills for different data roles
role_skills = analyzer.analyze_skills_by_role(pivot_df, top_n=12)

print("🎭 SKILL PROFILES BY JOB ROLE")
print("="*40)

# Visualize top roles
top_roles = list(role_skills.keys())[:6]  # Show first 6 roles

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, role in enumerate(top_roles):
    if idx < 6:
        ax = axes[idx]
        role_data = role_skills[role].head(8)
        
        colors = plt.cm.viridis(np.linspace(0, 1, len(role_data)))
        ax.barh(role_data['skill'], role_data['prevalence'], color=colors)
        ax.set_title(f'{role}', fontweight='bold', fontsize=12)
        ax.set_xlabel('Prevalence (%)')
        
        # Add value labels
        for i, v in enumerate(role_data['prevalence']):
            ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.suptitle('Top Skills by Data Science Role', fontsize=16, fontweight='bold', y=1.02)
plt.show()

### 4.2 Role Comparison Matrix

In [ ]:
# Compare skill profiles across roles
print("🔄 ROLE SKILL PROFILE COMPARISON")
print("="*45)

role_comparison = analyzer.compare_role_skill_profiles(pivot_df)

# Create heatmap for role comparison
plt.figure(figsize=(16, 10))

# Select top skills for visualization
top_skills_comparison = role_comparison.mean(axis=1).nlargest(15).index
comparison_vis = role_comparison.loc[top_skills_comparison]

# Create heatmap
sns.heatmap(comparison_vis, annot=True, fmt='.1f', cmap='YlOrRd', 
            linewidths=0.5, cbar_kws={'label': 'Prevalence (%)'})
plt.title('Skill Prevalence Heatmap Across Data Roles', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Job Role')
plt.ylabel('Skill')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n💡 Key Observations:")
print("• Roles with distinct skill profiles are clearly visible")
print("• Universal skills appear across multiple roles")
print("• Role-specific specializations are evident")

## 5. Trend Analysis
### 5.1 Emerging Skills Identification

In [ ]:

analyzer = DataScienceJobsAnalyzer(
    data_dir="../data",
    smoothing_alpha=0.3,
    significance_level=0.05,
    emerging_percentile=0.4,
    momentum_threshold=5
)

# ===== LOAD DATA =====
print("\n📁 Loading data...")
df = analyzer.load_cleaned_data()
print(f"✅ Loaded {len(df)} records")

# Create skill pivot table
pivot_df = analyzer.create_skill_pivot(df)
print(f"✅ Created pivot table with {len(pivot_df)} skill-month combinations")
print(f"📅 Date range: {pivot_df['date'].min()} to {pivot_df['date'].max()}")
print(f"🎯 Unique skills: {pivot_df['skill'].nunique()}")

# ===== RUN TREND ANALYSIS =====
print("\n📈 Running skill trend analysis...")
trend_results = analyzer.analyze_skill_trends_optimized(pivot_df, min_prevalence=1.0, min_months=6)

if trend_results.empty:
    print("❌ No trend results generated - check your data")
else:
    print(f"✅ Generated trend analysis for {len(trend_results)} skills")
    
    # Display summary
    category_counts = trend_results['trend_category'].value_counts()
    print(f"\n📊 Trend Category Distribution:")
    for category, count in category_counts.items():
        percentage = (count / len(trend_results)) * 100
        print(f"   • {category}: {count} skills ({percentage:.1f}%)")

# ===== VISUALIZATION 1: TREND CATEGORIES DISTRIBUTION =====
print("\n" + "="*50)
print("📊 VISUALIZATION 1: Trend Categories Distribution")
print("="*50)

fig1 = analyzer.plot_trend_categories_distribution(trend_results)
plt.suptitle('Data Science Skills - Trend Categories Distribution', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Display top skills by category
print("\n🔥 TOP EMERGING SKILLS:")
emerging_skills = analyzer.get_top_skills_by_category(trend_results, 'Emerging', 10)
if not emerging_skills.empty:
    for _, row in emerging_skills.iterrows():
        print(f"   • {row['skill']}: {row['recent_momentum_pct']:+.1f}% momentum, {row['current_prevalence']:.1f}% prevalence")
else:
    print("   No emerging skills found")

# ===== VISUALIZATION 2: EMERGING VS DECLINING SKILLS =====
print("\n" + "="*50)
print("📈 VISUALIZATION 2: Emerging vs Declining Skills")
print("="*50)

fig2 = analyzer.plot_emerging_vs_declining_skills(trend_results, top_n=15)
plt.suptitle('Top Emerging vs Declining Data Science Skills', fontsize=16, fontweight='bold', y=0.95)
plt.tight_layout()
plt.show()

# Career insights
print("\n🎯 CAREER INSIGHTS:")
emerging_count = trend_results['is_emerging'].sum()
declining_count = trend_results['is_declining'].sum()
print(f"   • {emerging_count} emerging skills → Focus on learning these")
print(f"   • {declining_count} declining skills → Avoid investing time in these")

# ===== VISUALIZATION 3: TREND SCATTER MATRIX =====
print("\n" + "="*50)
print("📊 VISUALIZATION 3: Trend Scatter Matrix")
print("="*50)

fig3 = analyzer.plot_trend_scatter_matrix(trend_results)
plt.suptitle('Data Science Skills - Trend Analysis Matrix', fontsize=16, fontweight='bold', y=0.95)
plt.tight_layout()
plt.show()

print("\n💡 INTERPRETATION GUIDE:")
print("   • Prevalence vs Slope: Top-right = high-demand growing skills")
print("   • CAGR vs Momentum: Consistent growth patterns")
print("   • Look for skills with high prevalence + positive slope")

# ===== VISUALIZATION 4: CORRELATION HEATMAP =====
print("\n" + "="*50)
print("🔗 VISUALIZATION 4: Correlation Heatmap")
print("="*50)

fig4 = analyzer.plot_heatmap_correlation(trend_results)
plt.suptitle('Skill Metrics Correlation Matrix', fontsize=16, fontweight='bold', y=0.95)
plt.tight_layout()
plt.show()

# ===== VISUALIZATION 5: TREND RADAR CHART =====
print("\n" + "="*50)
print("📡 VISUALIZATION 5: Trend Radar Chart")
print("="*50)

fig5 = analyzer.plot_trend_radar_chart(trend_results)
if fig5:
    plt.suptitle('Skill Categories - Radar Profile Comparison', fontsize=16, fontweight='bold', y=0.95)
    plt.tight_layout()
    plt.show()
else:
    print("   Not enough data for radar chart")

# ===== VISUALIZATION 6: SKILL TIMELINES =====
print("\n" + "="*50)
print("⏰ VISUALIZATION 6: Skill Timelines")
print("="*50)

# Get top 5 emerging skills for timeline
top_emerging_skills = analyzer.get_top_skills_by_category(trend_results, 'Emerging', 5)
if not top_emerging_skills.empty:
    skill_names = top_emerging_skills['skill'].tolist()
    fig6 = analyzer.plot_skill_trend_timeline(pivot_df, skill_names)
    plt.suptitle('Timeline: Top 5 Emerging Skills', fontsize=16, fontweight='bold', y=0.95)
    plt.tight_layout()
    plt.show()
else:
    print("   No emerging skills for timeline")

# ===== VISUALIZATION 7: INTERACTIVE DASHBOARD =====
print("\n" + "="*50)
print("🎮 VISUALIZATION 7: Interactive Dashboard")
print("="*50)

fig7 = analyzer.create_interactive_trend_dashboard(trend_results, pivot_df)
if fig7:
    fig7.show()
    print("✅ Interactive dashboard displayed - use hover for details!")
else:
    print("   Could not create interactive dashboard")

# ===== TREND SUMMARY AND INSIGHTS =====
print("\n" + "="*50)
print("📋 TREND SUMMARY AND INSIGHTS")
print("="*50)

# Get trend summary
summary = analyzer.get_trend_summary(trend_results)
if not summary.empty:
    print("\n📈 TREND CATEGORY SUMMARY:")
    print(summary)

# Generate actionable insights
print("\n🎯 ACTIONABLE INSIGHTS FOR DATA SCIENCE CAREERS:")

# Top emerging skills with good prevalence
high_potential_emerging = trend_results[
    (trend_results['is_emerging']) & 
    (trend_results['current_prevalence'] > 5)
].nlargest(5, 'recent_momentum_pct')

if not high_potential_emerging.empty:
    print(f"\n🔥 HIGH-POTENTIAL EMERGING SKILLS (learn now):")
    for _, skill in high_potential_emerging.iterrows():
        print(f"   • {skill['skill']}: {skill['recent_momentum_pct']:+.1f}% momentum, {skill['current_prevalence']:.1f}% prevalence")

# Established growing skills
established_growing = trend_results[
    (trend_results['is_growing']) & 
    (trend_results['current_prevalence'] > 20)
].nlargest(5, 'current_prevalence')

if not established_growing.empty:
    print(f"\n💼 ESTABLISHED GROWING SKILLS (solid investment):")
    for _, skill in established_growing.iterrows():
        print(f"   • {skill['skill']}: {skill['current_prevalence']:.1f}% prevalence, {skill['trend_slope']:.3f} slope")

# Declining skills to avoid
declining_to_avoid = trend_results[
    (trend_results['is_declining']) & 
    (trend_results['current_prevalence'] > 10)  # Still somewhat popular but declining
].nsmallest(5, 'recent_momentum_pct')

if not declining_to_avoid.empty:
    print(f"\n⚠️  DECLINING SKILLS (avoid investment):")
    for _, skill in declining_to_avoid.iterrows():
        print(f"   • {skill['skill']}: {skill['recent_momentum_pct']:+.1f}% momentum, {skill['current_prevalence']:.1f}% prevalence")

# ===== ADDITIONAL ANALYSIS: SKILL PROGRESSION =====
print("\n" + "="*50)
print("📊 ADDITIONAL ANALYSIS: Skill Progression by Seniority")
print("="*50)

# Analyze skill progression for top skills
top_skills_list = trend_results.nlargest(10, 'current_prevalence')['skill'].tolist()
progression_data = analyzer.analyze_skill_progression_data(pivot_df, top_skills_list)

if not progression_data.empty:
    analyzer.plot_skill_progression_organized(progression_data, max_skills=6)
else:
    print("   Not enough data for skill progression analysis")

# ===== EXPORT RESULTS =====
print("\n" + "="*50)
print("💾 EXPORTING RESULTS")
print("="*50)

# Save trend results
trend_results.to_csv('skill_trend_analysis_results.csv', index=False)
print("✅ Trend results saved to 'skill_trend_analysis_results.csv'")

# Save summary
if not summary.empty:
    summary.to_csv('trend_analysis_summary.csv')
    print("✅ Summary saved to 'trend_analysis_summary.csv'")

# ===== FINAL RECOMMENDATIONS =====
print("\n" + "="*50)
print("🎓 FINAL LEARNING RECOMMENDATIONS")
print("="*50)

print(f"""
Based on the analysis of {len(trend_results)} skills across your dataset:

🎯 IMMEDIATE FOCUS (0-3 months):
   • Learn {len(high_potential_emerging)} high-potential emerging skills
   • Master {len(established_growing)} established growing skills

📈 STRATEGIC FOCUS (3-12 months):
   • Monitor the emerging skills list quarterly
   • Develop projects combining emerging + established skills

⚠️  AVOID INVESTMENT:
   • {len(declining_to_avoid)} skills showing strong decline despite current popularity

💡 KEY SUCCESS FACTORS:
   • Balance learning emerging technologies with mastering established ones
   • Focus on skills with both growth potential AND current relevance
   • Regular skill assessment against market trends
""")

print("✅ NOTEBOOK EXECUTION COMPLETE!")

### 5.2 Declining Skills Analysis

In [ ]:
# Analyze declining skills
declining_skills = trends_report['declining_skills']
stable_skills = trends_report['stable_skills']

print("\n📉 SKILLS IN DECLINE")
print("-" * 30)

if not declining_skills.empty:
    plt.figure(figsize=(12, 6))
    
    declining_top = declining_skills.head(10)
    plt.barh(declining_top['skill'], declining_top['consensus_growth_rate'], color='red', alpha=0.7)
    plt.title('Top Declining Skills', fontsize=14, fontweight='bold')
    plt.xlabel('Growth Rate (%)')
    
    for i, v in enumerate(declining_top['consensus_growth_rate']):
        plt.text(v - 5, i, f'{v:.0f}%', va='center', fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.show()
    
    print("🔍 Declining Skills Details:")
    for _, skill in declining_skills.head(5).iterrows():
        print(f"   • {skill['skill']}: {skill['consensus_growth_rate']:.1f}% growth")
else:
    print("No significant declining trends detected")

## 6. Skill Ecosystem Analysis
### 6.1 Skill Combinations and Clusters

In [ ]:
# Analyze skill combinations and natural clusters
print("🔗 SKILL COMBINATIONS AND NATURAL CLUSTERS")
print("="*55)

# Analyze common skill combinations
skill_combinations = analyzer.analyze_skill_combination_prevalence(pivot_df, top_n=15)

if not skill_combinations.empty:
    print("\n🤝 TOP SKILL COMBINATIONS")
    print("-" * 30)
    
    # Display combination table
    combo_display = skill_combinations[['skill_1', 'skill_2', 'prevalence', 'combination_type', 'learning_synergy']]
    display(combo_display.head(10).style.background_gradient(cmap='Purples', subset=['prevalence'])
           .format({'prevalence': '{:.2f}%'}))
    
    # Visualize combinations
    plt.figure(figsize=(14, 8))
    top_combos = skill_combinations.head(12)
    
    # Create combination labels
    combo_labels = [f"{row['skill_1']} + {row['skill_2']}" for _, row in top_combos.iterrows()]
    
    plt.barh(combo_labels, top_combos['prevalence'], 
             color=plt.cm.Set3(np.linspace(0, 1, len(top_combos))))
    plt.xlabel('Prevalence (%)')
    plt.title('Most Common Skill Combinations in Job Postings', fontweight='bold')
    
    for i, v in enumerate(top_combos['prevalence']):
        plt.text(v + 0.01, i, f'{v:.2f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

### 6.2 Natural Skill Clusters

In [ ]:
# Identify natural skill clusters
print("\n🎯 NATURAL SKILL CLUSTERS")
print("-" * 30)

skill_clusters = analyzer.identify_natural_skill_clusters(pivot_df)

if skill_clusters:
    # Visualize clusters
    fig, ax = plt.subplots(figsize=(14, 8))
    
    cluster_names = list(skill_clusters.keys())
    cluster_sizes = [len(cluster['skills']) for cluster in skill_clusters.values()]
    cluster_co_occurrence = [cluster['avg_co_occurrence'] for cluster in skill_clusters.values()]
    
    # Create bubble chart
    scatter = ax.scatter(range(len(cluster_names)), cluster_co_occurrence, 
                        s=[size*100 for size in cluster_sizes], alpha=0.6, 
                        c=cluster_co_occurrence, cmap='viridis')
    
    # Add labels
    for i, (name, size) in enumerate(zip(cluster_names, cluster_sizes)):
        ax.annotate(f"{name}\n({size} skills)", 
                   (i, cluster_co_occurrence[i]), 
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=9, ha='left')
    
    ax.set_xlabel('Skill Clusters')
    ax.set_ylabel('Average Co-occurrence')
    ax.set_title('Natural Skill Clusters in Data Science Jobs', fontweight='bold')
    plt.colorbar(scatter, label='Co-occurrence Strength')
    plt.xticks([])
    plt.tight_layout()
    plt.show()
    
    # Print cluster details
    print("\n🔍 Cluster Details:")
    for cluster_name, cluster_data in list(skill_clusters.items())[:6]:
        print(f"\n   {cluster_name}:")
        print(f"      Core skill: {cluster_data['core_skill']}")
        print(f"      Skills: {', '.join(cluster_data['skills'][:4])}")
        print(f"      Avg co-occurrence: {cluster_data['avg_co_occurrence']:.1f}")

## 7. Regional and Company Analysis
### 7.1 Regional Skill Demand 

In [ ]:
# Analyze regional variations in skill demand
print("🌍 REGIONAL SKILL DEMAND VARIATIONS")
print("="*45)

regional_skills = analyzer.analyze_regional_skill_demand(pivot_df, top_n=8)

if regional_skills:
    # Plot top countries
    top_countries = list(regional_skills.keys())[:6]
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()
    
    for idx, country in enumerate(top_countries):
        if idx < 6:
            ax = axes[idx]
            country_data = regional_skills[country].head(6)
            
            ax.barh(country_data['skill'], country_data['prevalence'], 
                   color=plt.cm.tab10(idx))
            ax.set_title(f'{country}', fontweight='bold', fontsize=12)
            ax.set_xlabel('Prevalence (%)')
            
            for i, v in enumerate(country_data['prevalence']):
                ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.suptitle('Top Skills by Country/Region', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    print("💡 Regional Insights:")
    for country in top_countries[:3]:
        top_skill = regional_skills[country].iloc[0]['skill']
        prevalence = regional_skills[country].iloc[0]['prevalence']
        print(f"   • {country}: {top_skill} ({prevalence:.1f}%)")

## 8. Personalized Recommendations
### 8.1 Career Path Recommendations

In [ ]:
# Generate personalized skill recommendations
print("🎯 PERSONALIZED SKILL RECOMMENDATIONS")
print("="*50)

# Example: Data Scientist role at Mid-level
target_role = "Data Scientist"
target_seniority = "Mid-level"

personalized_recs = analyzer.generate_personalized_skill_recommendations(
    pivot_df, 
    target_role=target_role,
    target_seniority=target_seniority,
    top_n=15
)

if not personalized_recs.empty:
    print(f"\n💼 Recommended skills for {target_seniority} {target_role}:")
    print("-" * 55)
    
    plt.figure(figsize=(12, 8))
    analyzer.plot_skills_bar(personalized_recs, column='skill', top_n=12,
                           title=f'Top Skills for {target_seniority} {target_role}')
    
    # Display detailed recommendations
    display(personalized_recs[['skill', 'prevalence', 'mentions']]
           .head(10)
           .style.background_gradient(cmap='Blues', subset=['prevalence'])
           .format({'prevalence': '{:.1f}%', 'mentions': '{:,}'}))

### 8.2 Learning Path Generation

In [ ]:
# Create learning paths
print("\n🛣️  LEARNING PATH GENERATION")
print("="*40)

# Example: Current skills and target role
current_skills = ['Python', 'SQL', 'Machine Learning']
target_role = 'Data Scientist'

learning_path = analyzer.create_learning_paths(pivot_df, current_skills, target_role)

if learning_path:
    print(f"\n🎯 Current skills: {', '.join(learning_path['current_skills'])}")
    print(f"🎯 Target role: {target_role}")
    print(f"✅ Matching skills: {len(learning_path['matching_skills'])}")
    print(f"📚 Skills to learn: {len(learning_path['skill_gaps'])}")
    
    # Display skill gaps
    if learning_path['skill_gaps']:
        print("\n📖 RECOMMENDED LEARNING PATH:")
        print("-" * 35)
        
        gap_df = pd.DataFrame(learning_path['skill_gaps']).head(8)
        
        plt.figure(figsize=(10, 6))
        plt.barh(gap_df['skill'], gap_df['prevalence'], color='orange', alpha=0.7)
        plt.xlabel('Prevalence in Target Role (%)')
        plt.title(f'Top Skills to Learn for {target_role}', fontweight='bold')
        
        for i, v in enumerate(gap_df['prevalence']):
            plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # Display gap details
        display(gap_df[['skill', 'prevalence']]
               .style.background_gradient(cmap='Oranges', subset=['prevalence'])
               .format({'prevalence': '{:.1f}%'}))

## 9. Comprehensive Summary
### 9.1 Key Findings and Insights


In [ ]:
# Generate comprehensive summary
print("📊 COMPREHENSIVE ANALYSIS SUMMARY")
print("="*45)

# Calculate key metrics
total_jobs = pivot_df['job_ids'].explode().nunique()
total_skills = pivot_df['skill'].nunique()
avg_skills_per_job = pivot_df.groupby('job_ids').size().mean()

print(f"\n📈 KEY METRICS:")
print(f"   • Total Jobs Analyzed: {total_jobs:,}")
print(f"   • Unique Skills Identified: {total_skills:,}")
print(f"   • Average Skills per Job: {avg_skills_per_job:.1f}")

# Top insights
print(f"\n💡 TOP INSIGHTS:")
print(f"   1. Most demanded skill: {top_skills.iloc[0]['skill']} ({top_skills.iloc[0]['prevalence']:.1f}%)")
print(f"   2. Fastest growing category: {skill_categories.iloc[0]['skill_category']}")
print(f"   3. Most specialized role: {max(role_skills.items(), key=lambda x: len(x[1]))[0]}")

if not emerging_skills.empty:
    print(f"   4. Top emerging skill: {emerging_skills.iloc[0]['skill']} (+{emerging_skills.iloc[0]['consensus_growth_rate']:.0f}%)")

print(f"\n🎯 CAREER RECOMMENDATIONS:")
print(f"   • Focus on: {', '.join(top_skills['skill'].head(3).tolist())}")
print(f"   • Emerging areas: {', '.join(emerging_skills['skill'].head(2).tolist()) if not emerging_skills.empty else 'N/A'}")
print(f"   • Stable foundations: {', '.join(stable_skills['skill'].head(2).tolist()) if not stable_skills.empty else 'N/A'}")

print(f"\n✅ ANALYSIS COMPLETED SUCCESSFULLY!")